# Data Preprocessing for Modeling

In this notebook, we continue working on the Titanic data to predict survival by including more predictor variables. In particular, our code will:
- Handle missing values
- Encode categorical variables

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Colab only: mount Google Drive and set output folder for submissions
from google.colab import drive

drive.mount('/content/drive')

output_dir = '/content/drive/My Drive/Colab Notebook/output/'
os.makedirs(output_dir, exist_ok=True)
print(f'Submission files will be saved to: {output_dir}')

In [ ]:
# load dataset
df = pd.read_csv('https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/data/titanic_train.csv')
df.info()

In [ ]:
df.head()

In [ ]:
# PassengerId and name are not very useful for predicting 
# assume we cannot get useful information for prediction from ticket number 
# let's drop these columns
df.drop(['PassengerId', 'Name', 'Ticket'], axis = 1, inplace=True)
df.head()

## Handling Missing Values

There are two main types of approaches to handle missing values in data: 
- **Drop**
    - Drop instances with missing values
    - Drop attributes with missing values
- **Fill**
    - *Univariate Imputation*: filling the variable using values from the same variable, such as mean, median, most frequent, immediate before/after, etc.
    - *Multivariate Imputation*: imputing the variable's missing values via model-based approaches using other variables. (https://scikit-learn.org/stable/modules/impute.html)

In [ ]:
# total null values
df.isnull().sum()

In [ ]:
# given that only 2 out of 891 rows have missing values for Embarked - let's drop those two rows
# drop the observations/rows with missing values using dropna()
# note - we are not dropping the Embarked column!!
df.dropna(subset=['Embarked'], inplace=True)
df.head()

In [ ]:
# no missing values for Embarked
df.isnull().sum()

In [ ]:
# given 687 out of 891 rows have missing values for Cabin feature, let's drop this feature/column
# axis defaults to 0, which means row, here we are dropping the column, therefore axis=1
df.drop('Cabin', axis=1, inplace=True)
df.isnull().sum()

In [ ]:
# we cannot drop Age feature, which does not have as many as missing values as Cabin
df['Age'].hist(bins=50)

In [ ]:
df['Age'].describe()

In [ ]:
df['Age'].head(10)

In [ ]:
# method 1: fill the missing values with median using fillna()
# note that I don't use inplace=True because I want to demonstrate another method
median = df['Age'].median()
df_fill_median = df['Age'].fillna(median)
df_fill_median.head(10)

In [ ]:
# method 2: fill the missing values with median using Scikit-Learn SimpleImputer
# this approach can do imputation for all numerical attributes all at once
from sklearn.impute import SimpleImputer
median_imputer = SimpleImputer(strategy='median')

# select only numerical attributes for Imputer
# NOTE: Technically, Pclass (Ticket class with values 1, 2, or 3) is a categorical attribute.
# we treat Pclass as a numerical attribute here to keep it simple. 
df_num = df.select_dtypes(include=['int64','float64'])

# the following computes median for each attribute and stores the result in the built-in statistics_ variable
df_num_fill_median = median_imputer.fit_transform(df_num)  
median_imputer.statistics_  # same result as df_num.median().values

In [ ]:
df_num.median().values

In [ ]:
# imputer returns a numpy array
df_num_fill_median

In [ ]:
# change a numpy array to a DataFrame
df_num_fill_median = pd.DataFrame(df_num_fill_median, columns=df_num.columns)  
df_num_fill_median.isnull().sum()  # no missing values

## Processing the Target Column

Save the target column and drop it from the training set. Since the target column has no missing values now and we normally don't need to encode the target column (even if it has categorical values), let's save it and then drop it from the training set. 

In [ ]:
# set the target
y = df_num_fill_median['Survived']
df_num_fill_median.drop(['Survived'], axis=1, inplace=True)
df_num_fill_median.info()

In [ ]:
df_num_fill_median.describe()

Note that these numerical variables have very different distributions. 

In general, many learning algorithms benefit from standardization of the data set. 

There are different methods for standardization/scaling/normalization of data. 

For instance, we can scale features to lie between a given minimum and maximum value, often between zero and one. 

In [ ]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
df_num_fill_median_scaled = min_max_scaler.fit_transform(df_num_fill_median)

In [ ]:
df_num_fill_median_scaled

In [ ]:
# change df_num_fill_median_scaled into dataframe
df_num_processed = pd.DataFrame(df_num_fill_median_scaled, columns=df_num_fill_median.columns) 
df_num_processed

In [ ]:
df_num_processed.describe()

## Processing Categorical Variables 

So far, we have only included numerical variables in the model. Next, we will handle categorical variables. 

In [ ]:
# get categorical attributes
df_cat = df.select_dtypes(['object'])
df_cat.head()

In [ ]:
df_cat.describe()

In [ ]:
# encode categorical values to integers
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
df_cat_ordinal = ordinal_encoder.fit_transform(df_cat)
# categories are listed in order ['female', 'male']-> 0, 1; ['C', 'Q', 'S']-> 0, 1, 2
ordinal_encoder.categories_ 

In [ ]:
# encoder returns a numpy array
df_cat_ordinal

In [ ]:
# change df_cat_ordinal into dataframe
df_cat_ordinal = pd.DataFrame(df_cat_ordinal, columns=df_cat.columns) 
df_cat_ordinal

In [ ]:
# the same cat_encoder can be used to encode new data
new_data = [['male', 'S']]
new_data_encoded = ordinal_encoder.transform(new_data)
print(new_data_encoded)

## Build the model

Now we can prepare for the final train dataset and build the model:

- `df_num_processed`: df_num with missing values filled using median and scaled
- `df_cat_ordinal`: df_cat encoded using Ordinal Encoder

To use both numerical and categorical variables to build a classifier, we need to prepare the final training dataset by combining them. 

In [ ]:
# training dataset using ordinal encoder
titanic_train_encoded = pd.concat([df_num_processed, df_cat_ordinal], axis=1)
titanic_train_encoded.info()

In [ ]:
# train a DT model using titanic_train_encoded
from sklearn.tree import DecisionTreeClassifier
tree_clf1 = DecisionTreeClassifier(criterion='entropy', max_depth=3)
tree_clf1.fit(titanic_train_encoded, y)

## Make Predictions

When we train tree_clf1 above, titanic_train_encoded is the input dataframe we used. 

When we make predictions, the input dataframe we feed into the model **MUST** match the structure (the number of columns with specific order) of titanic_train_encoded.

In [ ]:
df_test = pd.read_csv('https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/data/titanic_test.csv')
df_test.info()

In [ ]:
# save PassengerId column for submission
df_test_id = df_test['PassengerId']

In [ ]:
df_test_id.head()

In [ ]:
# we dropped PassengerId, Name, Cabin, Ticket when training the model
# so we should drop those columns for prediction
df_test.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

In [ ]:
df_test.info()

In [ ]:
# we also need to handle missing values for the testing set
df_test.isnull().sum()

In [ ]:
# we use SimpleImputer to fill missing values in Age and Fare using median
median_imputer = SimpleImputer(strategy='median')

# select only numerical attributes for Imputer
df_test_num = df_test.select_dtypes(include=['int64','float64'])

# the following computes median for each attributes and store the result in statistics_ variable
df_test_num_fill_median = median_imputer.fit_transform(df_test_num)  

# convert df_test_num_fill_mean to dataframe
df_test_num_fill_median = pd.DataFrame(df_test_num_fill_median, columns=df_test_num.columns)  
df_test_num_fill_median.isnull().sum()  # no missing values

In [ ]:
# test numerical dataframe shape is 418x5
df_test_num_fill_median.info()

In [ ]:
# Check the distribution of df_test_num_fill_median
df_test_num_fill_median.describe()

In [ ]:
# Since the training set is scaled, we must use the same scaler for the test set too. 
# Important!!! Use .transform(), not .fit_transform()
df_test_num_fill_median_scaled = min_max_scaler.transform(df_test_num_fill_median)
df_test_num_fill_median_scaled

In [ ]:
# change df_test_num_fill_median_scaled into dataframe
df_test_num_processed = pd.DataFrame(df_test_num_fill_median_scaled, columns=df_test_num_fill_median.columns) 
df_test_num_processed

In [ ]:
# select only categorical attributes for Encoder
df_test_cat = df_test.select_dtypes(include=['object'])
df_test_cat.info()

In [ ]:
df_test_cat.head()

In [ ]:
# You must encode the test categorical dataframe 
# using the Ordinal Encoder we created for the training set
# DO NOT create a new encoder!!!

df_test_cat_ordinal = ordinal_encoder.transform(df_test_cat)

# convert df_test_cat_ordinal to dataframe
df_test_cat_ordinal = pd.DataFrame(df_test_cat_ordinal, columns=df_test_cat.columns)

In [ ]:
df_test_cat_ordinal.head()

In [ ]:
# combine the dataframes as the test input dataframe
titanic_test_encoded = pd.concat([df_test_num_processed, df_test_cat_ordinal], axis=1)
titanic_test_encoded.info()

In [ ]:
# the column array for the testing input dataframe using ordinal encoding
titanic_test_encoded.columns.values

In [ ]:
# the column array for the training input dataframe using ordinal encoding
titanic_train_encoded.columns.values

In [ ]:
titanic_test_encoded.head()

In [ ]:
# make prediction using the first tree
y_hat = tree_clf1.predict(titanic_test_encoded)
y_hat.shape

In [ ]:
# Note, the numbers here are float numbers with decimal, we need to convert them to integer
# if you don't do this, your score on Kaggle will be 0
y_hat

In [ ]:
y_hat = y_hat.astype(int)
y_hat

In [ ]:
# make the dataframe for submission by combining two columns
tree1_ordinal_submit = pd.DataFrame({
    'PassengerId': df_test_id, 
    'Survived': y_hat,
})

In [ ]:
tree1_ordinal_submit.head()

In [ ]:
# save the resulting dataframe as a csv file for Kaggle submission
tree1_ordinal_submit.to_csv(f'{output_dir}/tree1_ordinal_submit.csv', index=False) 

Let's pause for a second.  

**Is this ordinal encoder appropriate for encoding these two variables? Why or why not?**

In [ ]:
# Using one-hot encoding
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()
df_cat_onehot = onehot_encoder.fit_transform(df_cat)

onehot_encoder.categories_

In [ ]:
column_names = onehot_encoder.get_feature_names_out()
column_names

In [ ]:
# Take a look at df_cat_onehot
df_cat_onehot

In [ ]:
# onehot encoder returns a sparse matrix and we must convert that to a numpy array
# This step is different from ordinal encoding
df_cat_onehot = df_cat_onehot.toarray()

In [ ]:
# change df_cat_onehot to a Dataframe by adding the column names
df_cat_onehot = pd.DataFrame(df_cat_onehot, columns=column_names)
df_cat_onehot.head()

In [ ]:
# the same onehot_encoder can be used to encode new data
new_data1 = [['male', 'S']]
new_data1_encoded = onehot_encoder.transform(new_data1)
print(new_data1_encoded.toarray())

## Assignment 

Now, we have these two dataframes:  
- `df_num_processed`: df_num with missing values filled using median and scaled
- `df_cat_onehot`: df_cat encoded using OneHot Encoder

Following the same process above, you need to do the following: 
- Combine the two dataframes to make a new training set.
- Use the training set to build a new decision tree "tree_clf2".
- Apply "tree_clf2" to the test set with the two categorical variables 'Sex' and 'Embarked' encoded by the 'onehot_encoder'. 
- Prepare the prediction results in the right format: ['PassengerId', 'Survived']
- Name the output file "tree2_onehot_submit.csv" for submission
